In [ ]:
import cv2
import numpy as np

# Lade das Referenzbild und das Puzzleteil
reference_image_path = "../data/pzl_3/pzl_3_full_front.jpg"
puzzle_piece_path = "../data/pzl_3/pzl_3_003_front.jpg"

reference_image = cv2.imread(reference_image_path, cv2.IMREAD_COLOR)
puzzle_piece = cv2.imread(puzzle_piece_path, cv2.IMREAD_COLOR)

# Überprüfe, ob die Bilder erfolgreich geladen wurden
if reference_image is None:
    print(f"Error: Could not load reference image from {reference_image_path}")
    exit(1)

if puzzle_piece is None:
    print(f"Error: Could not load puzzle piece image from {puzzle_piece_path}")
    exit(1)

# Konvertiere die Bilder in Graustufen
reference_gray = cv2.cvtColor(reference_image, cv2.COLOR_BGR2GRAY)
puzzle_gray = cv2.cvtColor(puzzle_piece, cv2.COLOR_BGR2GRAY)

# Verwende ORB für die Merkmalsextraktion
orb = cv2.ORB_create()
keypoints1, descriptors1 = orb.detectAndCompute(puzzle_gray, None)
keypoints2, descriptors2 = orb.detectAndCompute(reference_gray, None)

# Führe den Merkmalsabgleich durch
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(descriptors1, descriptors2)
matches = sorted(matches, key=lambda x: x.distance)

# Zeichne die besten Übereinstimmungen
matched_image = cv2.drawMatches(puzzle_piece, keypoints1, reference_image, keypoints2, matches[:10], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

# Setze die Fenstergröße und zeige die Ergebnisse
cv2.namedWindow("Matches", cv2.WINDOW_NORMAL)  # Ermöglicht das Anpassen der Fenstergröße
cv2.resizeWindow("Matches", 800, 600)  # Setzt die Fenstergröße auf 800x600
cv2.imshow("Matches", matched_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Optional: Berechne die Transformation und die Position des Puzzleteils
if len(matches) > 4:
    src_pts = np.float32([keypoints1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([keypoints2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    matrix, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    h, w = puzzle_piece.shape[:2]
    corners = np.float32([[0, 0], [0, h], [w, h], [w, 0]]).reshape(-1, 1, 2)
    transformed_corners = cv2.perspectiveTransform(corners, matrix)

    # Zeichne die Position des Puzzleteils im Referenzbild
    reference_with_piece = reference_image.copy()
    cv2.polylines(reference_with_piece, [np.int32(transformed_corners)], True, (0, 255, 0), 3)

    cv2.namedWindow("Puzzle Piece Position", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Puzzle Piece Position", 800, 600)
    cv2.imshow("Puzzle Piece Position", reference_with_piece)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

[ WARN:0@0.022] global loadsave.cpp:268 findDecoder imread_('../data/pzl_3/pz3_1_full_front.jpg'): can't open/read file: check file path/integrity
[ WARN:0@0.023] global loadsave.cpp:268 findDecoder imread_('../data/pzl_3/pz3_1_003_front.jpg'): can't open/read file: check file path/integrity


Error: Could not load reference image from ../data/pzl_3/pz3_1_full_front.jpg
Error: Could not load puzzle piece image from ../data/pzl_3/pz3_1_003_front.jpg


error: OpenCV(4.11.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'
